# Getting started with v0.6

This self-contained example uses four notes, small semantic vectors and scripted
provider responses. It makes no network requests or model downloads. The names
demonstrate the pipeline API, not semantic quality. Documentation builds render
the cells without executing them.

For your own corpus, `Toponymy` defaults to `PLSCANClusterer` and exemplar-only
evidence. Here precomputed sparse cluster IDs make the output easy to inspect.


In [ ]:
import numpy as np
from toponymy import Toponymy, PrecomputedClusterer

objects = [
    "An orchard grows apples.",
    "The orchard harvest begins in autumn.",
    "A river flows toward the coast.",
    "Rain raises the river level.",
]
embedding_vectors = np.array([[1.0, 0.0], [0.9, 0.1], [0.0, 1.0], [0.1, 0.9]])
label_layers = [np.array([10, 10, 42, 42]), np.array([7, 7, 7, 7])]


In [ ]:
import json

class LocalNamer:
    """Scripted responses at the naming boundary, with no service calls."""
    def __init__(self):
        self.calls = 0

    def generate_topic_name(self, prompt, *, response_parser):
        self.calls += 1
        response = {"topic_name": f"Demo topic {self.calls}", "topic_specificity": 0.8}
        return response_parser(json.dumps(response))

    def generate_topic_cluster_names(self, prompt, old_names, *, response_parser):
        self.calls += 1
        response = {
            "new_topic_name_mapping": {
                str(i): f"{name} ({i})" for i, name in enumerate(old_names, 1)
            },
            "topic_specificities": [0.8] * len(old_names),
        }
        return response_parser(json.dumps(response))


## Prepare and inspect

`prepare` clusters, extracts evidence and renders initial `Prompt` objects
without calling the naming provider. The default exemplar extractor uses the
supplied semantic vectors; an additional text embedder is optional.


In [ ]:
namer = LocalNamer()
pipeline = Toponymy(
    namer,
    clusterer=PrecomputedClusterer(label_layers),
    object_description="notes",
    corpus_description="four short notes",
)
pipeline.prepare(objects, embedding_vectors)
assert namer.calls == 0
topic = pipeline.topics_[(0, 10)]
print(topic.features)
print(topic.prompt.system)
print(topic.prompt.user)


## Name topics

`name_topics()` completes the prepared run. `fit(objects, embedding_vectors,
clusterable_vectors=None)` combines preparation and naming. If a different
clustering representation is supplied, it may have a different dimension but
must have the same number of rows. Feature extraction still uses semantic vectors.


In [ ]:
pipeline.name_topics()
print(pipeline.topic_names_)
print(pipeline.topic_sizes_)
print(pipeline.topic_name_vectors_)
assert namer.calls == 3
assert set(pipeline.topics_) == {(0, 10), (0, 42), (1, 7)}


Each element of `topic_names_` and `topic_sizes_` is a dictionary
keyed by the original cluster ID. Noise (`-1`) creates no topic and appears as
`"Unlabelled"` in object-aligned `topic_name_vectors_`. Naming state lives on
topics, while cluster layers retain only membership.

## Next steps

Add evidence through [extractors](params_and_options.rst), configure
[summaries](topic_summaries.ipynb), or [save results](saving_loading.ipynb).
With an asynchronous wrapper, use `await pipeline.fit_async(...)` or
`await pipeline.name_topics_async()` after preparing. Every new fit recomputes
data-dependent stages. See the [migration guide](migration.rst) for v0.5 changes.
